### 1. Creación de la esquema 

In [1]:
from sqlalchemy import Date
from sqlalchemy import Column, Integer, String, Float
from sqlalchemy.orm import declarative_base

Base = declarative_base()

class Venta(Base):
    __tablename__ = 'ventas'
    id = Column(Integer, primary_key=True, autoincrement=True)
    fecha_venta = Column(Date, nullable=False)
    tipo_venta = Column(String, nullable=False)
    empleado = Column(String, nullable=False)
    categoria = Column(String, nullable=False)
    total = Column(Float, nullable=False)
    es_entregado = Column(Integer, nullable=False)  


### 2. Creacion de la DB local y población 

In [2]:
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from datetime import datetime
import pandas as pd

engine = create_engine('sqlite:///../db/ventas.db', echo=True, future=True)
Base.metadata.create_all(engine)

Session = sessionmaker(bind=engine)
session = Session()

try:
    path_file = "../data/ventas_imputado.csv"
    df_ventas = pd.read_csv(path_file, sep=",", encoding="utf-8")

    # Convertir fecha antes del insert
    df_ventas['fecha_venta'] = pd.to_datetime(df_ventas['fecha_venta']).dt.date

    # Bulk insert 
    registros = df_ventas[['fecha_venta', 'tipo_venta', 'empleado', 'categoria', 'total', 'es_entregado']].to_dict(orient='records')
    session.bulk_insert_mappings(Venta, registros)

    session.commit()
    print(f"{len(registros)} registros cargados correctamente.")

except Exception as e:
    session.rollback()  
    print(f" Error: {e}")

finally:
    session.close()

2026-04-27 12:02:57,280 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-27 12:02:57,280 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("ventas")
2026-04-27 12:02:57,281 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-27 12:02:57,282 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("ventas")
2026-04-27 12:02:57,283 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-27 12:02:57,285 INFO sqlalchemy.engine.Engine 
CREATE TABLE ventas (
	id INTEGER NOT NULL, 
	fecha_venta DATE NOT NULL, 
	tipo_venta VARCHAR NOT NULL, 
	empleado VARCHAR NOT NULL, 
	categoria VARCHAR NOT NULL, 
	total FLOAT NOT NULL, 
	es_entregado INTEGER NOT NULL, 
	PRIMARY KEY (id)
)


2026-04-27 12:02:57,286 INFO sqlalchemy.engine.Engine [no key 0.00080s] ()
2026-04-27 12:02:57,298 INFO sqlalchemy.engine.Engine COMMIT
2026-04-27 12:02:57,359 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-27 12:02:57,414 INFO sqlalchemy.engine.Engine INSERT INTO ventas (fecha_venta, tipo_venta, empleado, categor

In [3]:
from sqlalchemy import text


query = text("""
    SELECT id, fecha_venta, empleado, categoria, total, es_entregado
    FROM ventas
    WHERE tipo_venta = 'WhatsApp'
    ORDER BY total DESC
    LIMIT 10
""")

df_ventas_whatsapp = pd.read_sql_query(query, engine)

df_ventas_whatsapp

2026-04-27 12:03:04,898 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-27 12:03:04,899 INFO sqlalchemy.engine.Engine 
    SELECT id, fecha_venta, empleado, categoria, total, es_entregado
    FROM ventas
    WHERE tipo_venta = 'WhatsApp'
    ORDER BY total DESC
    LIMIT 10

2026-04-27 12:03:04,900 INFO sqlalchemy.engine.Engine [generated in 0.00239s] ()
2026-04-27 12:03:04,905 INFO sqlalchemy.engine.Engine ROLLBACK


,id,fecha_venta,empleado,categoria,total,es_entregado
0,3231,2023-02-07,Ana,hogar,2984.04,1
1,995,2023-01-01,Juan,tecnológico,2922.06,1
2,3897,2023-07-17,Juan,deportes,2862.24,1
3,1704,2023-06-29,Luis,ropa,2842.96,1
4,545,2023-08-07,Luis,tecnológico,2812.22,1
5,2595,2023-03-22,Luis,deportes,2725.10,1
6,977,2023-03-12,Sofía,muebles,2712.80,1
7,2430,2023-04-30,Ana,hogar,2636.63,1
8,3043,2023-06-14,Juan,ropa,2598.13,1
9,1011,2023-04-08,Sofía,ropa,2587.36,1
